# <center>Procesamiento Digital de Señales de Audio</center>
### <center>Instituto de Ingeniería Eléctrica - UdelaR</center>
## <center> Producto Armónico Espectral</center>

### Cómo correr este notebook

Es posible descargarlo y correrlo localmente en su computadora

Tambien pueden correrlo en Google Colab usando el siguiente link.

<table align="center">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/emidan19/audio-dsp/blob/main/practicos/P2a-short_time_GLogS.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Correr en Google Colab</a>
  </td>
</table>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import io, signal
import pandas as pd
from scipy.io.wavfile import read
from IPython.display import Audio, display

**NOTA:** *Las siguientes dos celdas solo son necesarias para descargar el archivo de ejemplo. Ignórelas si va a trabajar con sus propios archivos de audio.*

In [ ]:
!pip install wget

In [ ]:
import wget
from pathlib import Path

archivos = ['LP-mem-6-a.txt', 'LP-mem-6-a.wav']

# URL y archivo de salida
url = "https://github.com/emidan19/audio-dsp/blob/main/practicos/Archivos_P1/"
for archivo in archivos:
    # Descargar solo si no existe
    if not Path(archivo).exists():
        wget.download(url + archivo + "?raw=true")
        print(f"\nDescarga completada: {archivo}")
    else:
        print(f"El archivo ya existe: {archivo}")

# Log-Producto Armónico Espectral (logHPS)

En este ejercicio se estudia un método de detección de frecuencia fundamental (o _pitch_) basado en la transformada de Fourier de tiempo corto. Se sugiere utilizar el archivo *LP-mem-6-a.wav* para probarlo.

## Parte 1
El producto armónico espectral -Harmonic Product Spectrum, HPS- (ver _Pitch Detection in the Spectral Domain, pág. 623_, en  L. R. Rabiner and R. W. Schafer, _Theory and Applications of Digital Speech Processing_. Prentice Hall, 1st ed., 2011) está dado por

\begin{equation*}
	P_n(e^{jw}) = \prod_{r=1}^{K}|X_n(e^{jwr})|^2
\end{equation*}

Tomando el logaritmo se obtiene (log-Harmonic Product Spectrum, log-HPS),

\begin{equation*}
	\hat{P}_n(e^{jw}) = 2\sum_{r=1}^{K}\log|X_n(e^{jwr})|
\end{equation*}

**_Responder:_**

Explique por qué el HPS puede usarse para detección de _pitch_. Asuma que la señal de audio es monofónica (una única fuente armónica). ¿Qué ventajas presenta el uso del log-HPS frente al HPS? ¿Qué ocurre con señales cuya frecuencia fundamental está ausente (e.g. filtrado pasa-altos por el canal de comunicación)?


_Respuesta:_

...

## Parte 2

El espectro logarítmico acumulado (ó GLogS por sus siglas en inglés), se calcula como el promedio de logaritmo de la magnitud del espectro en posiciones armónicas de una frecuencia fundamental $f_0$, como

\begin{equation*}
	\rho_n(f_0) = \frac{1}{n_H}\sum_{i=1}^{n_H}\log|X_n(if_0)|
\end{equation*}

siendo $n_H$ la cantidad de armónicos de $f_0$ cuya frecuencia es menor a cierta frecuencia máxima $f_{\max}$.

Implemente un algoritmo de detección de _pitch_ que calcule el GLogS para valores de $f_0$ distribuidos de forma logarítmica entre $55Hz$ ($A1$) y $1046.5Hz$ ($C6$) con un paso de cuarto de tono, y $f_{\max} =  5000Hz$.

In [ ]:
def GLogS(Xn, fs, f0_grilla, fmax):
  '''
  Dada una señal de audio en su forma espectral, 
  calcula GLogS en el intervalo frecuencial dado 
  y con el maximo en frecuencia especificado.

  Entradas:
  Xn       : DFT de la señal/ventana de audio a calcular
  f0_grilla: Grilla de frecuencias a evaluar
  fmax     : Frecuencia máxima a tomar en el peso de la estimación GLogS
  fs       : Frecuencia de muestreo de la señal de audio
  '''
  # Tamaño de la DFT
  N = len(Xn)
  # Bins en frecuencia
  bins = np.arange(N)
  # Magnitud
  mag = np.abs(Xn)

  # Inicialización de la grilla de acumulación
  rho = np.zeros(len(f0_grilla))

  # Aplicar fuerza bruta (obtener acumulación para las frecuencias de la grilla)
  for i, f0 in enumerate(f0_grilla):

    # # COMPLETAR: cantidad de armónicos usados con f0
    nH = ...

    # COMPLETAR posiciones (no enteras) de los armónicos en bins de frecuencia
    k_arm = ...

    mag_arm = np.interp(k_arm, bins, mag)    # interpolación lineal

    # COMPLETAR: valor rho_f0 
    rho[i] = ...  
    
  return rho

## Parte 3

1. Completar el analisis en tiempo corto de trasformada de Fourier 

2. Utilice el GLogS para obtener una representación tiempo-$f_0$, que denominaremos $f_0$-grama. Compare dicha representación con el espectrograma usando el archivo de audio _LP-mem-6-a.wav_.

$\qquad$ _Nota:_  se recomienda un análisis DFT en tiempo corto con un salto de $0.01s$ para comparar con la $f_0$ de referencia.

3. **Responder:** ¿por qué tiene sentido utilizar $f_{max} = 5000~Hz$?

$\qquad$ _Nota:_ Puede ser de utilidad observar el espectrograma completo.

4. Represente la frecuencia fundamental detectada y la frecuencia de referencia en el $f_0$-grama en el archivo [_LP-mem-6-a.txt_](./Archivos_P3/LP-mem-6-a.txt). **Comente si aparecen los errores de armónicos y por qué**.

In [ ]:
fs, x = io.wavfile.read('LP-mem-6-a.wav')
Audio(x,rate=fs)

Implementación STFT

In [ ]:
def analysis_STFT(x, L=2048, Ra=256, win='hann'):
    """STFT de la señal de audio de entrada
    
    Parámetros
    ----------
    x : numpy array
        señal de audio de entrada (mono) como un arreglo numpy 1D.
    L : int
        longitud de la ventana en muestras.
    Ra : int
        salto entre ventanas en muestras.
    win : string
          tipo de ventana definido en scipy.signal.windows.
        
    Retorna
    -------
    X_stft : numpy array
             STFT de x como un arreglo numpy 2D.
    omega_stft : numpy array
                 valores de frecuencia en radianes.
    samps_stft : numpy array
                 muestra temporal correspondiente al comienzo de cada trama.

    """
    
    # longitud de la señal de entrada
    num_samps = x.size;      
    
    # ventana de análisis
    window = signal.windows.get_window(win, L)
   
    # número total de tramas de análisis
    num_frames = int(np.floor((num_samps - L) / Ra))

    # inicializar la STFT
    X_stft = np.zeros((L, num_frames), dtype = complex)
    
    # procesar cada trama
    for ind in range(num_frames):

        # puntos inicial y final de la trama
        n_ini = ... #COMPLETAR 
        n_end = ... #COMPLETAR

        # trama de la señal enventanada
        xr = ... #COMPLETAR

        # guardar la DFT de la trama de la señal
        X_stft[:, ind] = np.fft.fft(xr, n=L)
        
    # valores de frecuencia en radianes    
    omega_stft = 2*np.pi*np.linspace(0,1,L)

    # muestra temporal correspondiente al centro de cada trama
    samps_stft = np.arange(num_frames)*Ra+int(L/2)
 
    return X_stft, omega_stft, samps_stft

In [ ]:
# COMPLETAR: Obtener representacion en tiempo corto a saltos de 0.01s
X_stft, omega, t = ...

In [ ]:
# Visualizar espectrograma

# max frequency index
ind_fmax = int(X_stft.shape[0]/2)+1
# frequency values (Hz)
stft_freqs = omega[:ind_fmax]*fs/(2*np.pi)
# time values of the stft
stft_time = t/fs

plt.figure(figsize=(12,8))
ax1 = plt.subplot(2, 1, 1)
ax1.set_title('Espectrograma')
plt.pcolormesh(stft_time, stft_freqs, 20*np.log10(np.abs(X_stft[:ind_fmax, :])), cmap='jet')
plt.ylabel('Frecuencia (Hz)')
plt.xlabel('Tiempo (s)')
plt.ylim(0,stft_freqs[-1])
plt.show()

In [ ]:
def calcular_f0grama(X_short_time, fs, f0_grilla, fmax):
  # Cantidad de frames
  num_frames = X_short_time.shape[1]

  # Cantidad de frecuencias que tiene la grilla
  cant_f0 = len(f0_grilla)

  # Inicializar f0-grama y pitch en cada frame
  f0grama  = np.zeros((cant_f0,num_frames))
  f0s      = np.zeros(num_frames)

  # Indices y sus valores de las frecuencias con maximos en cada frame
  max_idxs = []
  max_vals = []

  # Construir f0-grama
  for t in np.arange(num_frames):
    rhos = GLogS(X_short_time[:,t], fs, f0_grilla, fmax)
    f0grama[:,t] = rhos

    # COMPLETAR: estimar la frecuencia fundamental del frame como la posición
    # del pico (máximo) de la acumulación GLogS (rhos)
    max_idxs.append(...)
    max_vals.append(...)

  # COMPLETAR: elegir un umbral basado en energia del f0-grama
  f0_energy_ths = ...


  # Asignar pitch si la señal supera el umbral de energia
  for t in np.arange(num_frames):
    # COMPLETAR: asignar el pitch estimado solo si supera la energia umbral
    if ...:
      f0s[t] = f0_grilla[max_idxs[t]]
    else:
      f0s[t] = 0

  return f0grama, f0s

Ejecutar GlogS y visualizar fo-grama

In [ ]:
f1, f2 = [55,1046.5]
steps = int(np.round(np.log2(f2/f1)*24))
# Se arma la grilla de frecuencias fundamentales a evaluar
f0_grilla = [f1]
for n in np.arange(steps):
  f0_grilla.append(f0_grilla[n]*2**(1/24))

# Frecuencia maxima 
f_max = 5000

# Calcular f0-grama y estimar frecuencias fundamentales
f0grama, f0s = calcular_f0grama(X_stft, fs, f0_grilla, f_max)

In [ ]:
plt.figure(figsize=(12,8))
ax1 = plt.subplot(2, 1, 1)
ax1.set_title('f0-grama')
plt.pcolormesh(np.arange(f0grama.shape[1]), f0_grilla, f0grama, cmap='jet')
plt.ylabel('Frecuencia (Hz)')
plt.xlabel('Frames')
plt.plot(np.arange(f0grama.shape[1]),f0s,'xk',alpha=0.4,label='Pitch detectado')
plt.ylim(0,1200)
plt.legend()
plt.show()

In [ ]:
### Se levanta el registro del pitch "ground truth"
gt_tiempo_pitch = pd.read_csv('LP-mem-6-a.txt', sep = "   ", names = ['tiempo','pitch'], header=None)
gt_tiempo = np.array(gt_tiempo_pitch['tiempo'])
f0s_gt  = np.array(gt_tiempo_pitch['pitch'])

In [ ]:
# Se hallan los errores respecto al pitch de referencia
f0_errs = np.abs((f0s-f0s_gt)/f0s_gt)

# Ubicar dónde hay errores mayores al 5%
samp_errs = np.argwhere(f0_errs > 0.05)
# Ubicar dónde ocurren errores de 2do armonico
harm2_errs = np.argwhere((f0_errs>1.9) & (f0_errs<2.1))
# Ubicar dónde ocurren errores de 3er armonico
harm3_errs = np.argwhere((f0_errs>2.85) & (f0_errs<3.15))

print(f'Errores > 5% (sin contar octava y 3er armónico): {len(samp_errs)-len(harm2_errs)-len(harm3_errs)}')

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(12,5))

ax1.set_title('Pitch en el tiempo')
ax1.plot(gt_tiempo, f0s_gt, 'b', label='Pitch referencia')
ax1.plot(gt_tiempo, f0s, 'k', label='Pitch detectado')
ax1.plot(gt_tiempo[samp_errs], f0s[samp_errs], 'or', label='Pitch errado')
ax1.plot(gt_tiempo[harm2_errs], f0s[harm2_errs], 'xg', label='Error 2do armónico')
ax1.plot(gt_tiempo[harm3_errs], f0s[harm3_errs], 'xy', label='Error 3er armónico')
ax1.set_ylabel('Pitch (Hz)')
ax1.legend()

plt.show()